In [1]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

# Find the project root that contains the src folder
current = Path.cwd().resolve()

for p in [current] + list(current.parents):
    if (p / "src").exists():
        repo_root = p
        break
else:
    raise FileNotFoundError("Could not find a folder containing 'src'.")

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

print("Current working directory:", current)
print("Added repo root:", repo_root)
print("src exists:", (repo_root / "src").exists())

from pathlib import Path
import numpy as np
import pandas as pd

import src.utils.pdata_io as pdio
from src.proc.extract_epoch_windows import load_epoch_windows

data_root, pdata_root, cc_data = pdio.load_project_context()

windows_df = load_epoch_windows(
    pdata_root=pdata_root,
    filename="behavior_epoch_windows.h5",
    key="windows/prepost_1s"
)

valid_windows = windows_df[windows_df["valid_window"]].copy()

print("All windows:", windows_df.shape)
print("Valid windows:", valid_windows.shape)

valid_windows.groupby(["phase", "epoch_name"]).size().reset_index(name="n_windows")

encoder_epoch_df = pd.read_hdf(
    Path(pdata_root) / "_cache" / "behavior_epoch_metrics.h5",
    key="encoder/prepost_1s_speedThresh_1cms"
)

from src.qc.qc_events import load_behavior_qc_tables

events_df, session_summary_df = load_behavior_qc_tables(
    pdata_root=pdata_root,
    filename="behavior_QC.h5"
)

from src.utils.pdata_organize import make_session_availability_summary

session_availability_df = make_session_availability_summary(
    events_df=events_df,
    windows_df=windows_df,
    min_session_duration_s=900,
    min_valid_events=3,
)

from src.utils.pdata_organize import add_day_bins_to_sessions

session_day_df = add_day_bins_to_sessions(
    session_availability_df,
    use_good_sessions_only=True,
)

Current working directory: /home/nmldata2/ccaw/Python/notebooks
Added repo root: /home/nmldata2/ccaw/Python
src exists: True
/home/nmldata2/ccaw/Python
[LOADED] Project context: /mnt/pdata/Classical_Conditioning/_cache/project_context.pkl
[LOADED] Epoch windows: /mnt/pdata/Classical_Conditioning/_cache/behavior_epoch_windows.h5
[KEY] windows/prepost_1s
All windows: (103980, 40)
Valid windows: (101861, 40)
[LOADED] Behavior QC tables: /mnt/pdata/Classical_Conditioning/_cache/behavior_QC.h5


In [3]:
# --------------------------------------------------
# Inspect phase and anchor names
# --------------------------------------------------

print("Phases:")
print(encoder_epoch_df["phase"].value_counts())

anchor_col = "anchor_name" if "anchor_name" in encoder_epoch_df.columns else "epoch_name"
print("\nUsing anchor column:", anchor_col)

print("\nAnchors by phase:")
display(
    encoder_epoch_df
    .groupby(["phase", anchor_col])
    .size()
    .reset_index(name="n")
    .sort_values(["phase", anchor_col])
)

print("\nWindow positions:")
print(encoder_epoch_df["window_position"].value_counts())

Phases:
phase
air_training         42362
habituation          38237
tone_air_training    21262
Name: count, dtype: int64

Using anchor column: anchor_name

Anchors by phase:


,phase,anchor_name,n
0,air_training,air_off,7060
1,air_training,air_off_mid,7033
2,air_training,air_on,7067
3,air_training,air_on_mid,7069
4,air_training,pseudo_tone_off,7070
5,air_training,pseudo_tone_on,7063
6,habituation,LED_off,6373
7,habituation,LED_off_mid,6327
8,habituation,LED_on,6390
9,habituation,LED_on_mid,6386



Window positions:
window_position
pre     50952
post    50909
Name: count, dtype: int64


In [4]:
# --------------------------------------------------
# Choose tone-air phase
# --------------------------------------------------

tone_phase_candidates = [
    p for p in encoder_epoch_df["phase"].dropna().unique()
    if "tone" in str(p).lower()
]

print("Tone phase candidates:", tone_phase_candidates)

tone_air_phase = tone_phase_candidates[0]
print("Using tone-air phase:", tone_air_phase)

Tone phase candidates: ['tone_air_training']
Using tone-air phase: tone_air_training


In [5]:
# --------------------------------------------------
# Prepare tone-air window dataframe
# --------------------------------------------------

tone_air_anchors = [
    "tone_on",
    "air_on",
    "tone_off",
    "air_on_mid",
    "air_off",
    "air_off_mid",
]

tone_air_epoch_map = {
    ("tone_on", "pre"): "pre_tone_on",
    ("tone_on", "post"): "post_tone_on",

    ("air_on", "pre"): "pre_air_on",
    ("air_on", "post"): "post_air_on",

    ("tone_off", "pre"): "pre_tone_off",
    ("tone_off", "post"): "post_tone_off",

    ("air_on_mid", "pre"): "pre_air_on_mid",
    ("air_on_mid", "post"): "post_air_on_mid",

    ("air_off", "pre"): "pre_air_off",
    ("air_off", "post"): "post_air_off",

    ("air_off_mid", "pre"): "pre_air_off_mid",
    ("air_off_mid", "post"): "post_air_off_mid",
}

tone_air_cycle_order = [
    "pre_tone_on",
    "post_tone_on",
    "pre_air_on",
    "post_air_on",
    "pre_tone_off",
    "post_tone_off",
    "pre_air_on_mid",
    "post_air_on_mid",
    "pre_air_off",
    "post_air_off",
    "pre_air_off_mid",
    "post_air_off_mid",
    "pre_tone_on_next",
]


def prepare_tone_air_window_df(
    encoder_epoch_df,
    phase,
    anchors,
    anchor_col="anchor_name",
    keep_overlap=True,
    require_good_session=True,
    require_valid_window=True,
):
    """
    Prepare long-format tone-air training window dataframe.

    One row = one pre/post window around a tone-air anchor.
    """

    df = encoder_epoch_df.copy()

    df = df[
        (df["phase"] == phase) &
        (df[anchor_col].isin(anchors)) &
        (df["window_position"].isin(["pre", "post"]))
    ].copy()

    if require_good_session and "good_session_basic" in df.columns:
        df = df[df["good_session_basic"] == True].copy()

    if require_valid_window and "valid_window" in df.columns:
        df = df[df["valid_window"] == True].copy()

    if not keep_overlap and "overlap_flag" in df.columns:
        df = df[df["overlap_flag"] == False].copy()

    # biological epoch labels
    df["tone_air_epoch_label"] = [
        tone_air_epoch_map.get((a, w), np.nan)
        for a, w in zip(df[anchor_col], df["window_position"])
    ]

    df = df[df["tone_air_epoch_label"].notna()].copy()

    # IDs
    df["animal_day"] = (
        df["animal"].astype(str) + ":" +
        df["date"].astype(str)
    )

    # Session time
    if "session_time_min" not in df.columns:
        if "anchor_session_time_min" in df.columns:
            df["session_time_min"] = df["anchor_session_time_min"]
        elif "anchor_time_s" in df.columns:
            df["session_time_min"] = df["anchor_time_s"] / 60.0
        elif "session_time_s" in df.columns:
            df["session_time_min"] = df["session_time_s"] / 60.0
        else:
            df["session_time_min"] = np.nan

    # Exposure session number
    if "phase_session_number" in df.columns:
        df["tone_air_session"] = df["phase_session_number"]
    elif "phase_day_number_good" in df.columns:
        df["tone_air_session"] = df["phase_day_number_good"]
    else:
        # Try merging from session_day_df
        merge_cols = [
            c for c in [
                "animal",
                "date",
                "phase",
                "phase_day_number_good",
                "phase_session_number",
            ]
            if c in session_day_df.columns
        ]

        sess_tmp = session_day_df[merge_cols].drop_duplicates().copy()

        df = df.merge(
            sess_tmp,
            on=[c for c in ["animal", "date", "phase"] if c in merge_cols],
            how="left",
            suffixes=("", "_sess")
        )

        if "phase_session_number" in df.columns:
            df["tone_air_session"] = df["phase_session_number"]
        elif "phase_day_number_good" in df.columns:
            df["tone_air_session"] = df["phase_day_number_good"]
        else:
            df["tone_air_session"] = np.nan

    return df.reset_index(drop=True)


tone_air_window_df = prepare_tone_air_window_df(
    encoder_epoch_df=encoder_epoch_df,
    phase=tone_air_phase,
    anchors=tone_air_anchors,
    anchor_col=anchor_col,
    keep_overlap=True,
    require_good_session=True,
    require_valid_window=True,
)

print(tone_air_window_df.shape)

display(
    tone_air_window_df["tone_air_epoch_label"]
    .value_counts()
    .reindex([x for x in tone_air_cycle_order if x != "pre_tone_on_next"])
)

(21154, 63)


tone_air_epoch_label
pre_tone_on         1766
post_tone_on        1766
pre_air_on          1766
post_air_on         1766
pre_tone_off        1766
post_tone_off       1765
pre_air_on_mid      1766
post_air_on_mid     1766
pre_air_off         1766
post_air_off        1759
pre_air_off_mid     1751
post_air_off_mid    1751
Name: count, dtype: int64

In [6]:
# --------------------------------------------------
# Build full tone-air cycle dataframe
# --------------------------------------------------

def make_tone_air_cycle_df(
    tone_air_window_df,
    trial_col="event_number",
    require_complete_cycles=True,
):
    """
    Build long-format full tone-air cycle dataframe.

    One cycle = trial N.

    Includes:
        all tone/air epochs from trial N
        plus pre_tone_on from trial N+1 as pre_tone_on_next
    """

    df = tone_air_window_df.copy()

    if trial_col not in df.columns:
        raise ValueError(f"{trial_col} not found in dataframe.")

    current_epochs = [
        "pre_tone_on",
        "post_tone_on",
        "pre_air_on",
        "post_air_on",
        "pre_tone_off",
        "post_tone_off",
        "pre_air_on_mid",
        "post_air_on_mid",
        "pre_air_off",
        "post_air_off",
        "pre_air_off_mid",
        "post_air_off_mid",
    ]

    # Current-trial epochs
    cur = df[df["tone_air_epoch_label"].isin(current_epochs)].copy()
    cur["cycle_trial"] = cur[trial_col].astype(int)
    cur["tone_air_cycle_epoch"] = cur["tone_air_epoch_label"]

    # Next pre-tone baseline:
    # pre_tone_on from trial N+1 becomes pre_tone_on_next for cycle N
    nxt = df[df["tone_air_epoch_label"] == "pre_tone_on"].copy()
    nxt["cycle_trial"] = nxt[trial_col].astype(int) - 1
    nxt["tone_air_cycle_epoch"] = "pre_tone_on_next"
    nxt = nxt[nxt["cycle_trial"] >= 1].copy()

    cycle_df = pd.concat([cur, nxt], ignore_index=True)

    # IDs
    cycle_df["animal_day"] = (
        cycle_df["animal"].astype(str) + ":" +
        cycle_df["date"].astype(str)
    )

    cycle_df["cycle_id"] = (
        cycle_df["animal"].astype(str) + ":" +
        cycle_df["date"].astype(str) + ":" +
        cycle_df["cycle_trial"].astype(str)
    )

    # Cycle-level time/session reference = pre_tone_on of that cycle
    cycle_info = (
        cycle_df[cycle_df["tone_air_cycle_epoch"] == "pre_tone_on"]
        [["animal", "date", "cycle_trial", "session_time_min", "tone_air_session"]]
        .drop_duplicates()
        .rename(columns={
            "session_time_min": "cycle_session_time_min",
            "tone_air_session": "cycle_tone_air_session",
        })
    )

    cycle_df = cycle_df.merge(
        cycle_info,
        on=["animal", "date", "cycle_trial"],
        how="left"
    )

    # Optional: keep only complete cycles
    if require_complete_cycles:
        counts = (
            cycle_df
            .groupby("cycle_id")["tone_air_cycle_epoch"]
            .nunique()
        )

        complete_cycle_ids = counts[counts == len(tone_air_cycle_order)].index

        cycle_df = cycle_df[
            cycle_df["cycle_id"].isin(complete_cycle_ids)
        ].copy()

    # Ordered categorical
    cycle_df["tone_air_cycle_epoch"] = pd.Categorical(
        cycle_df["tone_air_cycle_epoch"],
        categories=tone_air_cycle_order,
        ordered=True,
    )

    # Center predictors
    cycle_df["tone_air_session_c"] = (
        cycle_df["cycle_tone_air_session"] -
        cycle_df["cycle_tone_air_session"].mean()
    )

    cycle_df["cycle_session_10m_c"] = (
        cycle_df["cycle_session_time_min"] -
        cycle_df["cycle_session_time_min"].mean()
    ) / 10.0

    return cycle_df.reset_index(drop=True)


tone_air_cycle_df = make_tone_air_cycle_df(
    tone_air_window_df,
    trial_col="event_number",
    require_complete_cycles=True,
)

print(tone_air_cycle_df.shape)

display(
    tone_air_cycle_df["tone_air_cycle_epoch"]
    .value_counts()
    .sort_index()
)

(21762, 70)


tone_air_cycle_epoch
pre_tone_on         1674
post_tone_on        1674
pre_air_on          1674
post_air_on         1674
pre_tone_off        1674
post_tone_off       1674
pre_air_on_mid      1674
post_air_on_mid     1674
pre_air_off         1674
post_air_off        1674
pre_air_off_mid     1674
post_air_off_mid    1674
pre_tone_on_next    1674
Name: count, dtype: int64

In [7]:
# --------------------------------------------------
# Sanity checks
# --------------------------------------------------

print("Animals:", tone_air_cycle_df["animal"].nunique())
print("Animal-days:", tone_air_cycle_df["animal_day"].nunique())
print("Cycles:", tone_air_cycle_df["cycle_id"].nunique())
print("Observations:", len(tone_air_cycle_df))

display(
    tone_air_cycle_df[
        [
            "animal",
            "date",
            "cycle_trial",
            "tone_air_cycle_epoch",
            anchor_col,
            "window_position",
            "event_number",
            "cycle_tone_air_session",
            "cycle_session_time_min",
        ]
    ].head(40)
)

display(
    tone_air_cycle_df
    .groupby(["animal", "date"])["cycle_id"]
    .nunique()
    .reset_index(name="n_cycles")
    .head()
)

Animals: 5
Animal-days: 49
Cycles: 1674
Observations: 21762


,animal,date,cycle_trial,tone_air_cycle_epoch,anchor_name,window_position,event_number,cycle_tone_air_session,cycle_session_time_min
0,NML_04,2026_01_28,1,post_air_off_mid,air_off_mid,post,1,1.0,1.398697
1,NML_04,2026_01_28,1,pre_air_off_mid,air_off_mid,pre,1,1.0,1.398697
2,NML_04,2026_01_28,1,post_air_off,air_off,post,1,1.0,1.398697
3,NML_04,2026_01_28,1,pre_air_off,air_off,pre,1,1.0,1.398697
4,NML_04,2026_01_28,1,post_air_on_mid,air_on_mid,post,1,1.0,1.398697
5,NML_04,2026_01_28,1,pre_air_on_mid,air_on_mid,pre,1,1.0,1.398697
6,NML_04,2026_01_28,1,post_air_on,air_on,post,1,1.0,1.398697
7,NML_04,2026_01_28,1,pre_air_on,air_on,pre,1,1.0,1.398697
8,NML_04,2026_01_28,1,post_tone_off,tone_off,post,1,1.0,1.398697
9,NML_04,2026_01_28,1,pre_tone_off,tone_off,pre,1,1.0,1.398697


,animal,date,n_cycles
0,NML_04,2026_01_28,38
1,NML_04,2026_01_29,36
2,NML_04,2026_01_30,38
3,NML_04,2026_01_31,31
4,NML_04,2026_02_01,30


In [8]:
%load_ext rpy2.ipython

In [9]:
%%R -i tone_air_cycle_df -o tone_air_emmeans_R -o tone_air_contrasts_R -o tone_air_r2_R -o tone_air_fixed_R -o tone_air_session_trends_R -o tone_air_session_time_trends_R -o tone_air_model_info_R

library(lme4)
library(lmerTest)
library(emmeans)
library(broom.mixed)
library(dplyr)
library(performance)

# Use asymptotic df for emmeans because models are large
emm_options(lmer.df = "asymptotic")

# --------------------------------------------------
# Prepare factors
# --------------------------------------------------

tone_air_cycle_df$animal <- factor(tone_air_cycle_df$animal)
tone_air_cycle_df$animal_day <- factor(tone_air_cycle_df$animal_day)
tone_air_cycle_df$cycle_id <- factor(tone_air_cycle_df$cycle_id)

tone_air_cycle_df$tone_air_cycle_epoch <- factor(
  as.character(tone_air_cycle_df$tone_air_cycle_epoch),
  levels = c(
    "pre_tone_on",
    "post_tone_on",
    "pre_air_on",
    "post_air_on",
    "pre_tone_off",
    "post_tone_off",
    "pre_air_on_mid",
    "post_air_on_mid",
    "pre_air_off",
    "post_air_off",
    "pre_air_off_mid",
    "post_air_off_mid",
    "pre_tone_on_next"
  ),
  ordered = FALSE
)

# --------------------------------------------------
# Outcomes
# --------------------------------------------------

tone_air_outcomes_R <- c(
  "frac_moving",
  "frac_forward",
  "mean_speed_path_cms",
  "mean_speed_net_cms",
  "distance_path_cm",
  "distance_net_cm"
)

tone_air_outcomes_R <- tone_air_outcomes_R[
  tone_air_outcomes_R %in% names(tone_air_cycle_df)
]

print(tone_air_outcomes_R)

# --------------------------------------------------
# Planned adjacent contrasts
# 13 epochs, so each contrast vector has length 13
# --------------------------------------------------

tone_air_contrast_list <- list(
  tone_onset_transition =
    c(-1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0),

  tone_to_air_anticipatory_interval =
    c(0, -1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0),

  air_onset_transition =
    c(0, 0, -1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0),

  early_air_on_to_tone_offset =
    c(0, 0, 0, -1, 1, 0, 0, 0, 0, 0, 0, 0, 0),

  tone_offset_transition =
    c(0, 0, 0, 0, -1, 1, 0, 0, 0, 0, 0, 0, 0),

  post_tone_offset_air_on_progression =
    c(0, 0, 0, 0, 0, -1, 1, 0, 0, 0, 0, 0, 0),

  air_on_mid_transition =
    c(0, 0, 0, 0, 0, 0, -1, 1, 0, 0, 0, 0, 0),

  late_air_on_progression =
    c(0, 0, 0, 0, 0, 0, 0, -1, 1, 0, 0, 0, 0),

  air_offset_transition =
    c(0, 0, 0, 0, 0, 0, 0, 0, -1, 1, 0, 0, 0),

  early_air_off_recovery =
    c(0, 0, 0, 0, 0, 0, 0, 0, 0, -1, 1, 0, 0),

  air_off_mid_transition =
    c(0, 0, 0, 0, 0, 0, 0, 0, 0, 0, -1, 1, 0),

  late_air_off_recovery_to_next_baseline =
    c(0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, -1, 1)
)

# --------------------------------------------------
# Fit one outcome and extract tables
# --------------------------------------------------

fit_tone_air_cycle_outcome <- function(outcome_name) {

  cat("\n\n==============================\n")
  cat("Fitting outcome:", outcome_name, "\n")
  cat("==============================\n")

  formula_text <- paste0(
    outcome_name,
    " ~ tone_air_cycle_epoch * tone_air_session_c * cycle_session_10m_c + ",
    "(1 | animal) + (1 | animal_day) + (1 | cycle_id)"
  )

  m <- lmer(
    as.formula(formula_text),
    data = tone_air_cycle_df,
    REML = FALSE,
    control = lmerControl(
      optimizer = "bobyqa",
      optCtrl = list(maxfun = 2e5)
    )
  )

  model_info <- data.frame(
    outcome = outcome_name,
    n_obs = nobs(m),
    n_animals = nlevels(tone_air_cycle_df$animal),
    n_animal_day = nlevels(tone_air_cycle_df$animal_day),
    n_cycle_id = nlevels(tone_air_cycle_df$cycle_id),
    AIC = AIC(m),
    BIC = BIC(m),
    logLik = as.numeric(logLik(m)),
    singular = isSingular(m),
    stringsAsFactors = FALSE
  )

  emm <- emmeans(
    m,
    ~ tone_air_cycle_epoch,
    at = list(
      tone_air_session_c = 0,
      cycle_session_10m_c = 0
    ),
    lmer.df = "asymptotic"
  )

  emm_df <- as.data.frame(emm)
  emm_df$outcome <- outcome_name

  contrast_df <- as.data.frame(
    contrast(
      emm,
      tone_air_contrast_list,
      adjust = "none"
    )
  )
  contrast_df$outcome <- outcome_name

  r2_obj <- performance::r2_nakagawa(m)

  r2_df <- data.frame(
    outcome = outcome_name,
    R2_marginal = r2_obj$R2_marginal,
    R2_conditional = r2_obj$R2_conditional,
    stringsAsFactors = FALSE
  )

  fixed_df <- broom.mixed::tidy(
    m,
    effects = "fixed",
    conf.int = TRUE
  )
  fixed_df$outcome <- outcome_name

  session_trends <- emtrends(
    m,
    ~ tone_air_cycle_epoch,
    var = "tone_air_session_c",
    at = list(
      cycle_session_10m_c = 0
    ),
    lmer.df = "asymptotic"
  )

  session_trends_df <- as.data.frame(session_trends)
  session_trends_df$outcome <- outcome_name

  session_time_trends <- emtrends(
    m,
    ~ tone_air_cycle_epoch,
    var = "cycle_session_10m_c",
    at = list(
      tone_air_session_c = 0
    ),
    lmer.df = "asymptotic"
  )

  session_time_trends_df <- as.data.frame(session_time_trends)
  session_time_trends_df$outcome <- outcome_name

  return(
    list(
      model = m,
      model_info = model_info,
      emmeans = emm_df,
      contrasts = contrast_df,
      r2 = r2_df,
      fixed = fixed_df,
      session_trends = session_trends_df,
      session_time_trends = session_time_trends_df
    )
  )
}

# --------------------------------------------------
# Run all models
# --------------------------------------------------

tone_air_models_R <- list()
model_info_list <- list()
emmeans_list <- list()
contrasts_list <- list()
r2_list <- list()
fixed_list <- list()
session_trends_list <- list()
session_time_trends_list <- list()

for (outcome_name in tone_air_outcomes_R) {

  result <- fit_tone_air_cycle_outcome(outcome_name)

  tone_air_models_R[[outcome_name]] <- result$model
  model_info_list[[outcome_name]] <- result$model_info
  emmeans_list[[outcome_name]] <- result$emmeans
  contrasts_list[[outcome_name]] <- result$contrasts
  r2_list[[outcome_name]] <- result$r2
  fixed_list[[outcome_name]] <- result$fixed
  session_trends_list[[outcome_name]] <- result$session_trends
  session_time_trends_list[[outcome_name]] <- result$session_time_trends
}

# --------------------------------------------------
# Combine output tables
# --------------------------------------------------

tone_air_model_info_R <- bind_rows(model_info_list)
tone_air_emmeans_R <- bind_rows(emmeans_list)
tone_air_contrasts_R <- bind_rows(contrasts_list)
tone_air_r2_R <- bind_rows(r2_list)
tone_air_fixed_R <- bind_rows(fixed_list)
tone_air_session_trends_R <- bind_rows(session_trends_list)
tone_air_session_time_trends_R <- bind_rows(session_time_trends_list)

# --------------------------------------------------
# Reorder columns
# --------------------------------------------------

tone_air_emmeans_R <- tone_air_emmeans_R %>%
  relocate(outcome)

tone_air_contrasts_R <- tone_air_contrasts_R %>%
  relocate(outcome) %>%
  select(outcome, contrast, estimate, SE, df, z.ratio, p.value)

tone_air_fixed_R <- tone_air_fixed_R %>%
  relocate(outcome)

tone_air_session_trends_R <- tone_air_session_trends_R %>%
  relocate(outcome)

tone_air_session_time_trends_R <- tone_air_session_time_trends_R %>%
  relocate(outcome)

# --------------------------------------------------
# Print summaries
# --------------------------------------------------

cat("\n\n=== Tone-air model info ===\n")
print(tone_air_model_info_R)

cat("\n\n=== Tone-air adjacent contrasts ===\n")
print(tone_air_contrasts_R)

cat("\n\n=== Tone-air R2 ===\n")
print(tone_air_r2_R)

cat("\n\n=== Tone-air exposure-session trends by epoch ===\n")
print(tone_air_session_trends_R)

cat("\n\n=== Tone-air within-session-time trends by epoch ===\n")
print(tone_air_session_time_trends_R)

R[write to console]: Loading required package: Matrix

R[write to console]: 
Attaching package: ‘lmerTest’


R[write to console]: The following object is masked from ‘package:lme4’:

    lmer


R[write to console]: The following object is masked from ‘package:stats’:

    step


R[write to console]: Welcome to emmeans.
Caution: You lose important information if you filter this package's results.
See '? untidy'

R[write to console]: 
Attaching package: ‘dplyr’


R[write to console]: The following objects are masked from ‘package:stats’:

    filter, lag


R[write to console]: The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union




[1] "frac_moving"         "frac_forward"        "mean_speed_path_cms"
[4] "mean_speed_net_cms"  "distance_path_cm"    "distance_net_cm"    


Fitting outcome: frac_moving 


R[write to console]: NOTE: Results may be misleading due to involvement in interactions

R[write to console]: NOTE: Results may be misleading due to involvement in interactions

R[write to console]: NOTE: Results may be misleading due to involvement in interactions





Fitting outcome: frac_forward 


R[write to console]: NOTE: Results may be misleading due to involvement in interactions

R[write to console]: NOTE: Results may be misleading due to involvement in interactions

R[write to console]: NOTE: Results may be misleading due to involvement in interactions



...



Fitting outcome: distance_path_cm 


R[write to console]: NOTE: Results may be misleading due to involvement in interactions

R[write to console]: NOTE: Results may be misleading due to involvement in interactions

R[write to console]: NOTE: Results may be misleading due to involvement in interactions





Fitting outcome: distance_net_cm 


R[write to console]: NOTE: Results may be misleading due to involvement in interactions

R[write to console]: NOTE: Results may be misleading due to involvement in interactions

R[write to console]: NOTE: Results may be misleading due to involvement in interactions





=== Tone-air model info ===
              outcome n_obs n_animals n_animal_day n_cycle_id       AIC
1         frac_moving 21762         5           49       1674  4687.568
2        frac_forward 21762         5           49       1674  4238.129
3 mean_speed_path_cms 21762         5           49       1674 91855.438
4  mean_speed_net_cms 21762         5           49       1674 92756.242
5    distance_path_cm 21762         5           49       1674 91849.102
6     distance_net_cm 21762         5           49       1674 92781.938
        BIC     logLik singular
1  5134.892  -2287.784    FALSE
2  4685.453  -2063.065    FALSE
3 92302.762 -45871.719    FALSE
4 93203.565 -46322.121    FALSE
5 92296.425 -45868.551    FALSE
6 93229.262 -46334.969    FALSE


=== Tone-air adjacent contrasts ===
               outcome                               contrast     estimate
1          frac_moving                  tone_onset_transition  0.152895125
2          frac_moving      tone_to_air_anticipatory_i